### 12_UC_Tables_Views_Permissions

- Unity Catalog
- Unity Catalog Volumes 
- Managed Volumes 
- File in PySpark 
- Source Data in DBX

### Unity Catalog Tables 

- A unity catalog table is a table that is registered and goverened through unity catalog 
- We can query using SQL and Pyspark 
- Unity catalog manages the table metadata and access permissions
- Delta Lake provides transactional capabilities we have already seen it
- MERGE , UPDATE, INSERT DELETE


In [0]:
dbutils.widgets.text("catalog", "dataxdevelopment_1", "Unity Catalog")
catalog = dbutils.widgets.get("catalog")
if not catalog:
    raise ValueError("Parameter 'catalog' is required")
spark.sql(f"USE CATALOG `{catalog}`")
print(f"Using catalog: {catalog}")

Using catalog: dataxdevelopment_1


In [0]:
orders_path= f"/Volumes/{catalog}/default/datax/orders.csv"
print(orders_path)

/Volumes/dataxdevelopment_1/default/datax/orders.csv


In [0]:
from pyspark.sql.types import (    StructType, StructField,    IntegerType, StringType,    DateType, DecimalType)

orders_schema = StructType([    StructField("order_id", IntegerType()),    
                           StructField("customer_id", StringType()),    
                           StructField("product_id", StringType()),    
                           StructField("order_date", DateType()),    
                           StructField("quantity", IntegerType()),    
                           StructField("unit_price", DecimalType(10, 2)),    
                           StructField("order_status", StringType())])



In [0]:
orders_df = spark.read   .option("header", "true").schema(orders_schema).csv(orders_path)
display(orders_df)

order_id,customer_id,product_id,order_date,quantity,unit_price,order_status
1001,C101,P101,2026-09-01,2,899.00,COMPLETED
1002,C102,P103,2026-09-01,1,1499.00,COMPLETED
1003,C103,P102,2026-09-02,3,499.00,PENDING
1004,C101,P104,2026-09-02,1,2199.00,CANCELLED
1005,C104,P101,2026-09-03,2,899.00,COMPLETED
1006,C105,P105,2026-09-03,1,3499.00,COMPLETED
1007,C102,P103,2026-09-04,2,1499.00,PENDING
1008,C106,P106,2026-09-04,4,299.00,COMPLETED
1009,C107,P102,2026-09-05,1,499.00,COMPLETED
1010,C103,P104,2026-09-05,2,2199.00,COMPLETED


In [0]:
orders_df.write.format("delta").mode("overwrite").saveAsTable('bronze.orders_csv')

In [0]:
%sql
Select * from bronze.orders_csv

order_id,customer_id,product_id,order_date,quantity,unit_price,order_status
1001,C101,P101,2026-09-01,2,899.00,COMPLETED
1002,C102,P103,2026-09-01,1,1499.00,COMPLETED
1003,C103,P102,2026-09-02,3,499.00,PENDING
1004,C101,P104,2026-09-02,1,2199.00,CANCELLED
1005,C104,P101,2026-09-03,2,899.00,COMPLETED
1006,C105,P105,2026-09-03,1,3499.00,COMPLETED
1007,C102,P103,2026-09-04,2,1499.00,PENDING
1008,C106,P106,2026-09-04,4,299.00,COMPLETED
1009,C107,P102,2026-09-05,1,499.00,COMPLETED
1010,C103,P104,2026-09-05,2,2199.00,COMPLETED


In [0]:
%sql
Describe detail bronze.orders_csv

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,f067e753-8e9a-471e-84fa-dc3aee7b43d8,dataxdevelopment_1.bronze.orders_csv,null,abfss://unity-catalog-storage@dbstoragewe57fmye274fa.dfs.core.windows.net/7405605023169443/__unitystorage/catalogs/96d0da40-33f6-4533-9bb7-9d10eb755e03/tables/3e25a440-03c9-47ed-9f93-3b8f684da66c,2026-10-27T14:24:41.495Z,2026-10-27T14:24:42Z,List(),List(),1,2891,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 3e25a440-03c9-47ed-9f93-3b8f684da66c, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-9bd82c60-a4b0-4269-9371-8c20ab6d7085, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-243c2b81-ffcb-4e57-a2f0-ed5ebd1b6a66)",3,7,"List(appendOnly, clustering, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",true


In [0]:
%sql
Select * from bronze.orders_csv order by order_id desc

order_id,customer_id,product_id,order_date,quantity,unit_price,order_status
1010,C103,P104,2026-09-05,2,2199.00,COMPLETED
1009,C107,P102,2026-09-05,1,499.00,COMPLETED
1008,C106,P106,2026-09-04,4,299.00,COMPLETED
1007,C102,P103,2026-09-04,2,1499.00,PENDING
1006,C105,P105,2026-09-03,1,3499.00,COMPLETED
1005,C104,P101,2026-09-03,2,899.00,COMPLETED
1004,C101,P104,2026-09-02,1,2199.00,CANCELLED
1003,C103,P102,2026-09-02,3,499.00,PENDING
1002,C102,P103,2026-09-01,1,1499.00,COMPLETED
1001,C101,P101,2026-09-01,2,899.00,COMPLETED


# UNITY Catalog Views

- A view is a saved SQL Query that presentws data from one or more tables 
- A standard view doesnt store a sperate copy of the query result 
- Whenever we execute or query it , then the underlying quey is executed and you see the result
- Views are useful for reuabale busienss logic and controlled access of the data


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS GOLD

In [0]:
%sql
CREATE OR REPLACE VIEW GOLD.vw_orders_completed as
Select 
order_id,
customer_id,
product_id,
order_date, 
quantity, 
unit_price,
quantity*unit_price as Total_Amount
from bronze.orders_csv Where order_status='COMPLETED'

In [0]:
%sql
Select * from GOLD.vw_orders_completed

order_id,customer_id,product_id,order_date,quantity,unit_price,Total_Amount
1001,C101,P101,2026-09-01,2,899.00,1798.00
1002,C102,P103,2026-09-01,1,1499.00,1499.00
1005,C104,P101,2026-09-03,2,899.00,1798.00
1006,C105,P105,2026-09-03,1,3499.00,3499.00
1008,C106,P106,2026-09-04,4,299.00,1196.00
1009,C107,P102,2026-09-05,1,499.00,499.00
1010,C103,P104,2026-09-05,2,2199.00,4398.00


# USING COMMAND GRAND REVOKE ACCESS ON UNITY CATALOG

In [0]:
%sql 
SHOW GRANTS ON CATALOG ${catalog}

Principal,ActionType,ObjectType,ObjectKey
_workspace_users_dataxdev_7405605023169443,USE CATALOG,CATALOG,dataxdevelopment_1


In [0]:
%sql
GRANT USE SCHEMA 
ON SCHEMA ${catalog}.gold
TO `account users`